[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-09-information-flow-control.ipynb)

# Advanced Discussion 9 — information-flow control and the three-capabilities check

A small policy engine that labels data and checks every tool call, applied to eight scenarios and four agent designs.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Pure Python; runs instantly.

In [ ]:
from dataclasses import dataclass, field

## 1. label every piece of data with where it came from, and check every tool call against the labels

In [ ]:
@dataclass(frozen=True)
class Label:
    secret: bool = False                    # confidential: customer records, credentials, internal documents
    untrusted: bool = False                 # may contain instructions from an attacker: web pages, inbound email, uploaded files
def join(*labels): return Label(secret=any(l.secret for l in labels), untrusted=any(l.untrusted for l in labels))

@dataclass
class Tool:
    name: str
    external: bool = False                  # sends data outside the trust boundary (email, HTTP POST, public ticket)
    side_effect: bool = False               # changes something (write, refund, delete)

@dataclass
class Session:
    seen_private: bool = False              # has this session ever read private data?
    seen_untrusted: bool = False            # ... untrusted content?
    log: list = field(default_factory=list)

def decide(session, tool, arg_labels):
    """Return ('allow' | 'ask' | 'deny', reason). arg_labels: the labels of every value that flows into the call."""
    lab = join(*arg_labels) if arg_labels else Label()
    if lab.secret and tool.external:                                    return "deny", "secret data must not leave the boundary"
    if lab.untrusted and tool.side_effect:                              return "ask", "an action shaped by untrusted content needs a human"
    if session.seen_private and session.seen_untrusted and tool.external: return "deny", "private data + untrusted content + external channel in one session"
    if tool.external and session.seen_untrusted:                        return "ask", "external send after reading untrusted content"
    return "allow", "no rule triggered"

PUBLIC, INTERNAL, WEB, SECRET_WEB = Label(), Label(secret=True), Label(untrusted=True), Label(secret=True, untrusted=True)
read_kb, web_fetch = Tool("search_kb"), Tool("web_fetch", external=True)
send_email, write_note, refund = Tool("send_email", external=True), Tool("write_note", side_effect=True), Tool("issue_refund", side_effect=True, external=False)
scenarios = [
 ("summarise a public FAQ",                       Session(),                                   read_kb,    [PUBLIC]),
 ("email a public FAQ answer to the customer",   Session(),                                   send_email, [PUBLIC]),
 ("email a customer's account details",           Session(seen_private=True),                  send_email, [INTERNAL]),
 ("write a note based on a fetched web page",    Session(seen_untrusted=True),                write_note, [WEB]),
 ("issue a refund shaped by an inbound email",   Session(seen_untrusted=True),                refund,     [WEB, PUBLIC]),
 ("fetch a URL after reading private records",   Session(seen_private=True, seen_untrusted=True), web_fetch, [PUBLIC]),
 ("email a summary after browsing the web",      Session(seen_untrusted=True),                send_email, [PUBLIC]),
 ("post secret text extracted from a web page",   Session(seen_private=True, seen_untrusted=True), send_email, [SECRET_WEB]),
]
print("%-45s %-8s %s" % ("scenario", "verdict", "reason"))
for name, sess, tool, labels in scenarios:
    v, why = decide(sess, tool, labels); print("%-45s %-8s %s" % (name, v, why))

## 2. the 'three capabilities' check for a whole agent design, before you write any prompt

In [ ]:
def trifecta(reads_private, reads_untrusted, can_exfiltrate):
    n = sum([reads_private, reads_untrusted, can_exfiltrate]); return "HIGH RISK: all three present" if n == 3 else f"{n} of 3 present"
print()
for name, caps in (("FAQ bot (public docs only, no tools)", (False, False, False)), ("support agent: CRM + email inbox + reply email", (True, True, True)),
                   ("research agent: web browsing + notes (no private data)", (False, True, False)), ("internal analyst: private DB + no external tools", (True, False, False))):
    print("%-58s -> %s" % (name, trifecta(*caps)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.